# Multi Marginal Optimal Transport

In [1]:
import random
import torch
import numpy as np
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from torch.amp import GradScaler
from torch.utils.data import DataLoader
import deeplake
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score
from sklearn.decomposition import PCA
from geomloss import SamplesLoss
import time
import pandas as pd
from torch.utils.data import ConcatDataset, RandomSampler
from torch.optim.lr_scheduler import CosineAnnealingLR
import pickle
import seaborn as sns
import itertools
from dataset_OT import make_multi_WSI_dataset

seed = 42
torch.manual_seed(seed)
np.random.seed(42)
random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False 



In [2]:
#data loading


batch_size = 64 

print('Loading starting...')
idx_range_subset1 = [i for i in range(1,52+1)]
random.shuffle(idx_range_subset1)
num_train = int(np.ceil(0.7 * len(idx_range_subset1))) 
train_range1, val_range1 = idx_range_subset1[:num_train], idx_range_subset1[num_train:]

idx_range_subset3 = [i for i in range(1,26+1)] 
random.shuffle(idx_range_subset3)
num_train = int(np.ceil(0.7 * len(idx_range_subset3))) 
train_range3, val_range3 = idx_range_subset3[:num_train], idx_range_subset3[num_train:]

akoya_loader_train_subset1 = make_multi_WSI_dataset('Subset1', train_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset1 = make_multi_WSI_dataset('Subset1', val_range1, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_train_subset3 = make_multi_WSI_dataset('Subset3', train_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
akoya_loader_val_subset3 = make_multi_WSI_dataset('Subset3', val_range3, ['Akoya'], train_or_test='Train', batch_size=batch_size)
leica_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
leica_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['Leica'], train_or_test='Train', batch_size=batch_size)
kfbio_loader_train = make_multi_WSI_dataset('Subset3', train_range3, ['KFBio'], train_or_test='Train', batch_size=batch_size)
kfbio_loader_val = make_multi_WSI_dataset('Subset3', val_range3, ['KFBio'], train_or_test='Train', batch_size=batch_size)

akoya_loader_train = ConcatDataset([akoya_loader_train_subset1, akoya_loader_train_subset3])
akoya_loader_val = ConcatDataset([akoya_loader_val_subset1, akoya_loader_val_subset3])

len_akoya_train = len(akoya_loader_train)
len_akoya_val = len(akoya_loader_val)

len_leica_train = len(leica_loader_train)
len_leica_val = len(leica_loader_val)

len_kfbio_train = len(kfbio_loader_train)
len_kfbio_val = len(kfbio_loader_val)

len_train = len_akoya_train + len_leica_train + len_kfbio_train
len_val = len_akoya_val + len_leica_val + len_kfbio_val

B_A_train = round(batch_size * len_akoya_train / (len_akoya_train + len_leica_train))
B_L_train = batch_size - B_A_train 
B_A_val = round(batch_size * len_akoya_val / (len_akoya_val + len_leica_val))
B_L_val = batch_size - B_A_val

akoya_loader_train = DataLoader(akoya_loader_train, batch_size=B_A_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
akoya_loader_val = DataLoader(akoya_loader_val, batch_size=B_A_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_train = DataLoader(leica_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
leica_loader_val = DataLoader(leica_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
kfbio_loader_train = DataLoader(kfbio_loader_train, batch_size=B_L_train, shuffle=True, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)
kfbio_loader_val = DataLoader(kfbio_loader_val, batch_size=B_L_val, num_workers=6, pin_memory=True, persistent_workers=True, prefetch_factor=4)

print("Train batches Akoya:", len(akoya_loader_train), 'batch size:', B_A_train)
print("Train batches Leica:", len(leica_loader_train), 'batch size:', B_L_train)
print("Train batches KFBio:", len(kfbio_loader_train), 'batch size:', B_L_train)
print("Validation batches Akoya:", len(akoya_loader_val), 'batch size:', B_A_val)
print("Validation batches Leica:", len(leica_loader_val), 'batch size:', B_L_val)

#in samples
print('len train:', len_train)

Loading starting...
Train batches Akoya: 39681 batch size: 50
Train batches Leica: 40173 batch size: 14
Train batches KFBio: 48926 batch size: 14
Validation batches Akoya: 10072 batch size: 49
Validation batches Leica: 10467 batch size: 15
len train: 3231403


In [3]:
import torch

def pairwise_sqdist(x, y):
    # x: (n,d), y: (m,d)
    x2 = (x**2).sum(dim=1).unsqueeze(1)  # (n,1)
    y2 = (y**2).sum(dim=1).unsqueeze(0)  # (1,m)
    xy = x @ y.T                      # (n,m)
    return x2 + y2 - 2*xy             # (n,m)

def make_broadcast_shape(sizes, axis):
    shape = [1] * len(sizes)
    shape[axis] = sizes[axis]
    return tuple(shape)

def multimarginal_sinkhorn_torch(as_list, C, eps=0.1, n_iters=50, atol=1e-12, renormalize_u=True):
    """
    Multimarginal Sinkhorn in PyTorch.

    as_list: list of 1D torch tensors (probability marginals, sum=1)
    C: cost tensor (shape matches sizes of as_list)
    eps: entropic regularization parameter
    n_iters: number of iterations
    """
    S = len(as_list)
    sizes = [a.shape[0] for a in as_list]

    # Kernel
    K = torch.exp(-C / eps)

    # Scaling vectors
    u_list = [torch.ones(n, device=C.device, dtype=C.dtype) for n in sizes]
    b_shapes = [make_broadcast_shape(sizes, s) for s in range(S)]

    for _ in range(n_iters):
        for s in range(S):
            Q = K.clone()
            for r in range(S):
                if r == s: 
                    continue
                Q *= u_list[r].reshape(b_shapes[r])

            axes_to_sum = tuple(i for i in range(S) if i != s)
            m_s = Q.sum(dim=axes_to_sum)
            m_s = torch.clamp(m_s, min=atol)

            u_new = as_list[s] / m_s
            if renormalize_u:
                u_new = u_new / (u_new.sum() + atol)
            u_list[s] = u_new

    P = K.clone()
    for s in range(S):
        P *= u_list[s].reshape(b_shapes[s])

    P = P / (P.sum() + atol)
    ot_cost = torch.sum(P * C)
    return ot_cost, u_list


In [4]:
# Test

sup_times = []


tq = tqdm(zip(akoya_loader_train, leica_loader_train, kfbio_loader_train),
                                                    desc=f"Timing OT losses",
                                                    total=min(len(akoya_loader_train), len(leica_loader_train), len(kfbio_loader_train)))

for batch_akoya, batch_leica, batch_kfbio in tq:
    X = batch_akoya['embedding']   # (n_a, d)
    Y = batch_leica['embedding']   # (n_l, d)
    Z = batch_kfbio['embedding']   # (n_k, d)

    # uniform marginals
    t0 = time.time()
    a = torch.full((X.shape[0],), 1/X.shape[0], device=X.device)
    b = torch.full((Y.shape[0],), 1/Y.shape[0], device=Y.device)
    c = torch.full((Z.shape[0],), 1/Z.shape[0], device=Z.device)

    # cost tensor
    C_xy = pairwise_sqdist(X, Y)
    C_xz = pairwise_sqdist(X, Z)
    C_yz = pairwise_sqdist(Y, Z)
    C = C_xy.unsqueeze(2) + C_xz.unsqueeze(1) + C_yz.unsqueeze(0)
    C = C / C.max()

    #print("C min:", C.min().item(), "C max:", C.max().item())
    ot_loss, _ = multimarginal_sinkhorn_torch([a, b, c], C, eps=0.1, n_iters=20)
    t1 = time.time()

    sup_times.append(t1 - t0)
    avg_time = sum(sup_times) / len(sup_times)
    tq.set_description(f"Timing OT losses | Avg time: {avg_time:.3f}s | MMOT: {ot_loss.item():.4f}")
    
        

Timing OT losses | Avg time: 0.004s | MMOT: 0.6905:   2%|▋                                      | 733/39681 [07:13<6:24:18,  1.69it/s]


KeyboardInterrupt: 

Bad pipe message: %s [b'K\xd6)\x9b\xacq\xbd~']
Bad pipe message: %s [b'<\x9ap\xfb\xe5(\xac_\x00\x02\xbc\x00\x00\x00\x01\x00\x02\x00\x03\x00\x04\x00\x05\x00\x06\x00\x07\x00\x08\x00\t\x00\n\x00\x0b\x00\x0c\x00\r\x00\x0e\x00\x0f\x00\x10\x00\x11\x00\x12\x00\x13\x00\x14\x00\x15\x00\x16\x00\x17\x00\x18\x00\x19\x00\x1a\x00\x1b\x00\x1e\x00\x1f\x00 \x00!\x00"\x00#\x00$\x00%\x00&\x00\'\x00(\x00)\x00*\x00+\x00,\x00-\x00.\x00/\x000\x001\x002\x003\x004\x005\x006\x007\x008\x009\x00:\x00;\x00<\x00=\x00>\x00?\x00@\x00A\x00B\x00C\x00D\x00E\x00F\x00g\x00h\x00i\x00j\x00k\x00l\x00m\x00\x84\x00\x85\x00\x86\x00\x87\x00\x88\x00\x89\x00\x8a\x00\x8b\x00\x8c\x00\x8d\x00\x8e\x00\x8f\x00\x90\x00\x91\x00\x92\x00\x93\x00\x94\x00\x95\x00\x96\x00\x97\x00\x98\x00\x99\x00\x9a\x00\x9b\x00\x9c\x00\x9d\x00\x9e\x00']
Bad pipe message: %s [b'\xa0\x00\xa1\x00\xa2\x00\xa3\x00\xa4\x00\xa5\x00\xa6\x00\xa7\x00\xa8\x00\xa9\x00\xaa\x00\xab\x00\xac\x00\xad\x00\xae\x00\xaf\x00\xb0\x00\xb1\x00\xb2\x00\xb3\x00\xb4\x00\xb5\x00\xb6\x00\